# SRGAN Training

This notebook trains SRGAN for 32 x 32 to 128 x 128 OCT super-resolution.

- Uses training split from `model_a.ipynb`
- Saves checkpoints every n epochs.

In [ ]:
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# Locate the project root directory.def find_root():
    cwd = Path.cwd().resolve()
    for root in [cwd, *cwd.parents]:
        if (root / "notebooks").exists() and (root / "requirements.txt").exists():
            return root
    for root in [Path("/content/OCT-SRGAN"), Path("/content/drive/MyDrive/OCT-SRGAN")]:
        if root.exists():
            return root
    return cwd

root_dir = find_root()
train_csv = root_dir / "artifacts" / "splits" / "train_split.csv"
if not train_csv.exists():
    raise FileNotFoundError(
        "train_split.csv was not found. Run notebooks/model_a.ipynb first to create split files."
    )

ckpt_dir = root_dir / "checkpoints" / "srgan"
ckpt_dir.mkdir(parents=True, exist_ok=True)

print("root_dir:", root_dir)

train_df = pd.read_csv(train_csv)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Torch version:", torch.__version__)
print("CUDA ready:", torch.cuda.is_available())
print("Train samples:", len(train_df))

In [ ]:
high_size = 128
low_size = 32
batch_size = 32
epochs = 15
save_gap = 5
worker_n = 2
adv_w = 1e-3
feat_w = 1.0
pix_w = 0.01

In [ ]:
# Dataset for low/high-resolution training pairs.class SRSet(Dataset):
    # Initialize class state.    def __init__(self, data_df):
        self.data_df = data_df.reset_index(drop=True)
        self.to_img = transforms.ToTensor()
        self.high_tf = transforms.Resize(
            (high_size, high_size),
            interpolation=transforms.InterpolationMode.BICUBIC,
        )

    # Return the dataset size.    def __len__(self):
        return len(self.data_df)

    # Return a single sample.    def __getitem__(self, idx):
        img = Image.open(self.data_df.iloc[idx]["path"]).convert("RGB")
        high_img = self.to_img(self.high_tf(img))
        low_img = F.interpolate(high_img.unsqueeze(0), size=(low_size, low_size), mode="area").squeeze(0)
        return low_img, high_img

train_set = SRSet(train_df)
train_loader = DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True,
    num_workers=worker_n,
    pin_memory=True,
)

plt.figure(figsize=(12, 8))
for idx in range(4):
    low_img, high_img = train_set[np.random.randint(0, len(train_set))]
    low_show = F.interpolate(low_img.unsqueeze(0), size=(high_size, high_size), mode="nearest").squeeze(0)
    plt.subplot(4, 2, 2 * idx + 1)
    plt.imshow(high_img.permute(1, 2, 0).numpy().clip(0, 1))
    plt.title("High 128 x 128")
    plt.axis("off")
    plt.subplot(4, 2, 2 * idx + 2)
    plt.imshow(low_show.permute(1, 2, 0).numpy().clip(0, 1))
    plt.title("Low 32 x 32 upsampled")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Residual block used in SRGAN models.class ResBlock(nn.Module):
    # Initialize class state.    def __init__(self, channels=64):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, 1, 1),
            nn.BatchNorm2d(channels),
            nn.PReLU(),
            nn.Conv2d(channels, channels, 3, 1, 1),
            nn.BatchNorm2d(channels),
        )

    # Run the forward pass.    def forward(self, x):
        return x + self.block(x)

# Generator network for super-resolution.class GenNet(nn.Module):
    # Initialize class state.    def __init__(self, blocks=8):
        super().__init__()
        self.head = nn.Sequential(nn.Conv2d(3, 64, 9, 1, 4), nn.PReLU())
        self.body = nn.Sequential(*[ResBlock(64) for _ in range(blocks)])
        self.mid = nn.Sequential(nn.Conv2d(64, 64, 3, 1, 1), nn.BatchNorm2d(64))
        self.up = nn.Sequential(
            nn.Conv2d(64, 256, 3, 1, 1),
            nn.PixelShuffle(2),
            nn.PReLU(),
            nn.Conv2d(64, 256, 3, 1, 1),
            nn.PixelShuffle(2),
            nn.PReLU(),
        )
        self.tail = nn.Sequential(nn.Conv2d(64, 3, 9, 1, 4), nn.Sigmoid())

    # Run the forward pass.    def forward(self, x):
        skip_feat = self.head(x)
        body_feat = self.body(skip_feat)
        mix_feat = self.mid(body_feat) + skip_feat
        return self.tail(self.up(mix_feat))

# Feature block for the discriminator.class DiscBlock(nn.Module):
    # Initialize class state.    def __init__(self, in_ch, out_ch, stride):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride, 1),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    # Run the forward pass.    def forward(self, x):
        return self.block(x)

# Discriminator network for adversarial learning.class DiscNet(nn.Module):
    # Initialize class state.    def __init__(self):
        super().__init__()
        self.feat = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),
            DiscBlock(64, 64, 2),
            DiscBlock(64, 128, 1),
            DiscBlock(128, 128, 2),
            DiscBlock(128, 256, 1),
            DiscBlock(256, 256, 2),
            DiscBlock(256, 512, 1),
            DiscBlock(512, 512, 2),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 1),
        )

    # Run the forward pass.    def forward(self, x):
        return self.head(self.feat(x))

gen_net = GenNet().to(device)
disc_net = DiscNet().to(device)

feat_net = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features[:36].to(device).eval()
for param in feat_net.parameters():
    param.requires_grad = False

adv_loss = nn.BCEWithLogitsLoss()
pix_loss = nn.L1Loss()
gen_opt = optim.Adam(gen_net.parameters(), lr=1e-4, betas=(0.9, 0.999))
disc_opt = optim.Adam(disc_net.parameters(), lr=1e-4, betas=(0.9, 0.999))
img_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
img_std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

In [ ]:
fix_low, fix_high = next(iter(train_loader))
fix_low = fix_low[:4].to(device)
fix_high = fix_high[:4].to(device)

# Preview low, generated, and target images.def show_view(epoch):
    gen_net.eval()
    with torch.no_grad():
        super_img = gen_net(fix_low).clamp(0, 1).cpu()
        low_show = F.interpolate(fix_low.cpu(), size=(high_size, high_size), mode="nearest")
        high_img = fix_high.cpu()
    fig, axis = plt.subplots(4, 3, figsize=(10, 12))
    for idx in range(4):
        axis[idx, 0].imshow(low_show[idx].permute(1, 2, 0).numpy().clip(0, 1))
        axis[idx, 0].axis("off")
        axis[idx, 0].set_title("Low-Resolution Input")
        axis[idx, 1].imshow(super_img[idx].permute(1, 2, 0).numpy().clip(0, 1))
        axis[idx, 1].axis("off")
        axis[idx, 1].set_title("Generated Super-Resolution")
        axis[idx, 2].imshow(high_img[idx].permute(1, 2, 0).numpy().clip(0, 1))
        axis[idx, 2].axis("off")
        axis[idx, 2].set_title("Real High-Resolution")
    fig.tight_layout()
    fig.suptitle(f"SRGAN preview at epoch {epoch:03d}", y=1.02)
    plt.show()
    plt.close(fig)

log_rows = []
for epoch in range(1, epochs + 1):
    time0 = time.time()
    gen_net.train()
    disc_net.train()
    gen_hist, disc_hist = [], []
    feat_hist, pix_hist, adv_hist = [], [], []
    for low_img, high_img in train_loader:
        low_img = low_img.to(device, non_blocking=True)
        high_img = high_img.to(device, non_blocking=True)
        batch_n = high_img.size(0)
        real_lab = torch.ones((batch_n, 1), device=device)
        fake_lab = torch.zeros((batch_n, 1), device=device)
        with torch.no_grad():
            fake_img = gen_net(low_img).detach()
        disc_val = adv_loss(disc_net(high_img), real_lab) + adv_loss(disc_net(fake_img), fake_lab)
        disc_opt.zero_grad(set_to_none=True)
        disc_val.backward()
        disc_opt.step()
        super_img = gen_net(low_img)
        adv_val = adv_loss(disc_net(super_img), real_lab)
        feat_val = pix_loss(
            feat_net((super_img - img_mean) / img_std),
            feat_net((high_img - img_mean) / img_std),
        )
        pix_val = pix_loss(super_img, high_img)
        gen_val = (adv_w * adv_val) + (feat_w * feat_val) + (pix_w * pix_val)
        gen_opt.zero_grad(set_to_none=True)
        gen_val.backward()
        gen_opt.step()
        gen_hist.append(float(gen_val.item()))
        disc_hist.append(float(disc_val.item()))
        feat_hist.append(float(feat_val.item()))
        pix_hist.append(float(pix_val.item()))
        adv_hist.append(float(adv_val.item()))

    row = {
        "epoch": epoch,
        "gen_loss": np.mean(gen_hist),
        "disc_loss": np.mean(disc_hist),
        "feat_loss": np.mean(feat_hist),
        "pix_loss": np.mean(pix_hist),
        "adv_loss": np.mean(adv_hist),
        "sec": time.time() - time0,
    }
    log_rows.append(row)

    print(
        "Epoch {:03d}/{} | gen_loss={:.4f} disc_loss={:.4f} sec={:.1f}".format(
            epoch,
            epochs,
            row["gen_loss"],
            row["disc_loss"],
            row["sec"],
        )
    )

    if epoch % save_gap == 0:
        torch.save(gen_net.state_dict(), ckpt_dir / f"generator_epoch_{epoch:03d}.pth")
        torch.save(disc_net.state_dict(), ckpt_dir / f"discriminator_epoch_{epoch:03d}.pth")
        show_view(epoch)

torch.save(gen_net.state_dict(), ckpt_dir / "generator_final.pth")
torch.save(disc_net.state_dict(), ckpt_dir / "discriminator_final.pth")

log_df = pd.DataFrame(log_rows)
fig, axis = plt.subplots(1, 2, figsize=(12, 4))
axis[0].plot(log_df["epoch"], log_df["gen_loss"], label="Generator")
axis[0].plot(log_df["epoch"], log_df["disc_loss"], label="Discriminator")
axis[0].set_title("Adversarial Losses")
axis[0].set_xlabel("Epoch")
axis[0].legend()

axis[1].plot(log_df["epoch"], log_df["feat_loss"], label="Feature")
axis[1].plot(log_df["epoch"], log_df["pix_loss"], label="Pixel")
axis[1].set_title("Reconstruction Losses")
axis[1].set_xlabel("Epoch")
axis[1].legend()

fig.tight_layout()
plt.show()